# glasses-twin inference run

Runs the offline pipeline: capture.mp4 -> frames -> VGGT poses -> 3DGRUT splat.

Inputs:
- `harrishayy21/glasses-twin-bundle` — wheels + VGGT + 3DGRUT + weights + repo code
- `harrishayy21/test-scene` — input video (`test_scene.mp4`)

Outputs land in `/kaggle/working/scene/` and are auto-saved as kernel output.

In [ ]:
# Cell 1 - GPU + env sanity. Bail early if the GPU is too old (P100 sm_60 won't work with PyTorch 2.10+).
import subprocess, sys, os
print('python:', sys.version.split()[0])
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv']).decode())
import torch
print('torch:', torch.__version__, '| cuda:', torch.version.cuda, '| cudnn:', torch.backends.cudnn.version())
assert torch.cuda.is_available(), 'no GPU available'

cap = torch.cuda.get_device_capability(0)
name = torch.cuda.get_device_name(0)
print(f'GPU: {name} (sm_{cap[0]}{cap[1]})')
if cap[0] < 7:
    raise RuntimeError(
        f'GPU {name} (sm_{cap[0]}{cap[1]}) is too old for the installed PyTorch '
        f'(needs sm_70+). Switch the kernel accelerator to "GPU T4 x2" '
        f'(sm_75) in the Kaggle notebook UI before re-running.'
    )

In [ ]:
# Cell 2 - Install bundle (wheels + VGGT + 3DGRUT + repo code). 5-15 min.
# Kaggle preserves the `bundle/` prefix from the upload, so the real root may be one level deeper.
import os, subprocess
_DATASET_ROOT = '/kaggle/input/glasses-twin-bundle'
BUNDLE = _DATASET_ROOT + '/bundle' if os.path.isdir(_DATASET_ROOT + '/bundle') else _DATASET_ROOT
print('BUNDLE =', BUNDLE)
# subprocess.run (not !bash) so a non-zero exit aborts the cell instead of silently
# letting later cells try to import packages that never installed.
subprocess.run(['bash', f'{BUNDLE}/install.sh', BUNDLE], check=True)

In [ ]:
# Cell 3 - Set up the scene directory and extract frames from input video
import os, subprocess, json
from pathlib import Path

SCENE_ID = os.environ.get('SCENE_ID', 'kaggle_v0')
ARTIFACTS = Path(f'/kaggle/working/artifacts')
SCENE = ARTIFACTS / 'scenes' / SCENE_ID
SCENE.mkdir(parents=True, exist_ok=True)
(SCENE / 'frames').mkdir(exist_ok=True)

os.environ['ARTIFACTS_PATH'] = str(ARTIFACTS)
os.environ['VGGT_LOCAL_WEIGHTS'] = f'{BUNDLE}/weights/vggt.pt'
os.environ['THREEDGRUT_ROOT'] = f'{BUNDLE}/src/3dgrut'

video = '/kaggle/input/test-scene/test_scene.mp4'
assert os.path.exists(video), f'no input video at {video} — attach harrishayy21/test-scene to the kernel'

# Use the capture module to write frames + capture.yaml + initial manifest.
# We need an initial manifest first since capture/inference both UPDATE one rather than create.
from datetime import datetime, timezone
from shared.schemas import Manifest, Stages, Stage, Artifacts, Stats
base = f'/artifacts/scenes/{SCENE_ID}'
Manifest(
    scene_id=SCENE_ID,
    created_at=datetime.now(timezone.utc),
    status='processing',
    stages=Stages(
        capture=Stage(status='pending'),
        poses=Stage(status='pending'),
        splat=Stage(status='pending'),
        segmentation=Stage(status='pending'),
    ),
    artifacts=Artifacts(
        splat_ply=f'{base}/splat.ply',
        annotations_json=f'{base}/annotations.json',
        thumbnail_jpg=f'{base}/thumbnail.jpg',
        cameras_json=f'{base}/cameras.json',
    ),
    stats=Stats(frame_count=0, object_count=0, splat_size_mb=0.0),
).write_atomic(SCENE / 'manifest.json')

# Extract frames at 2 fps
subprocess.run(['ffmpeg', '-y', '-i', video, '-vf', 'fps=2', f'{SCENE}/frames/%04d.png'], check=True)
n = len(list((SCENE / 'frames').glob('*.png')))
print(f'extracted {n} frames -> {SCENE}/frames/')

In [ ]:
# Cell 4 - Run the inference CLI for real (VGGT + 3DGRUT)
import subprocess
subprocess.run([
    'python', '-m', 'inference',
    '--scene-id', SCENE_ID,
    '--scene-dir', str(SCENE),
    '--real',
    '--iterations', '7000',
], check=True)

In [ ]:
# Cell 5 - Verify outputs
import os, json
for name in ['cameras.json', 'points.ply', 'splat.ply', 'manifest.json']:
    p = SCENE / name
    print(f'  {name}: {p.stat().st_size if p.exists() else "MISSING"} bytes')

manifest = json.loads((SCENE / 'manifest.json').read_text())
print('\nstages:')
for k, v in manifest['stages'].items():
    print(f'  {k}: {v}')

In [ ]:
# Cell 6 - Pack outputs into a single archive for easy download
import shutil
out = '/kaggle/working/scene_output'
shutil.make_archive(out, 'zip', SCENE)
print(f'wrote {out}.zip ({os.path.getsize(out + ".zip") / 1e6:.1f} MB)')